# SkillRadar – Del 7: Udviklingsproces, kvalitet, design, privacy, deployment og kritisk projektforsvar
### Ekstra dybdegående eksamensnotebook · 5. november 2026

**Formål.** Denne notebook uddyber de seks områder, som ligger mellem de tekniske beregninger og det samlede projektforsvar. Den er bygget på jeres egne GitHub-filer og skrevet til den mundtlige eksamen på DTU 02830.

**Læsevejledning**
- **Dokumenteret:** Fundet i projektets filer. Kilde og filnavn er angivet.
- **Fortolkning:** Analytisk forklaring på hvad beslutningen betyder; ikke en ny observation fra jeres målinger.
- **Muligt fremtidigt arbejde:** Forslag, ikke noget I allerede har gennemført.
- Kodecellerne er **selvstændige små eksempler** til forståelse og må ikke forveksles med genkørsel af jeres fulde GitHub-repository.

Alle kodeeksempler er lavet til standard Python, pandas og NumPy. De ændrer ingen projektfiler.

## Indholdsfortegnelse
1. Projektets udviklingsproces og beslutninger.
2. Testing, debugging og softwarekvalitet.
3. Human-Centred Design og brugertest.
4. Privacy, GDPR og sikkerhed.
5. Deployment, dependencies og reproducerbarhed.
6. Kritisk evaluering af projektets evidens.
7. Eksamensspørgsmål og et sammenhængende projektforsvar.

**Kildehierarki:** Konkrete resultater hentes fra `evaluation/results/*.md` og `analysis/results/goal2_output.txt`; udviklingsstatus fra `TODO.md` (opdateret 4. november 2026); implementering fra Python-filer; dokumenterede procedurer fra `evaluation/` og `user_testing/`. Når README og TODO omtaler en plan, behandles det som en plan – ikke som en færdig test.

# A. Projektets udviklingsproces og beslutninger
## A1. Fra problem til tre testbare mål

**Dokumenteret i `README.md`:** Projektets målgruppe er studerende, som søger praktik eller deres første job. Spørgsmålet er ikke blot »hvad kan mit CV?«, men »hvilke *navngivne teknologier* nævner arbejdsmarkedets opslag for en valgt rolle, og hvilke nævnes ikke i mit CV?«.

I opdelte projektet i **tre mål**:

| Mål | Spørgsmål | Artefakt | Hvordan I måler det |
|---|---|---|---|
| Goal 1 | Kan navngivne teknologier udtrækkes? | O*NET keyword-matcher, LLM, hybrid | Menneskelige labels, precision, recall, F1, ablation |
| Goal 2 | Er demand-ranking stabil? | Posting→tools-tabeller og statistik | Spearman, bootstrap af rang, junior/senior-test |
| Goal 3 | Er prototypen forståelig og brugbar? | Streamlit-app og testversion | Think-aloud, opgavegennemførelse, correction rate, spørgeskema |

**Hvorfor er opdelingen vigtig?** En præcis extractor garanterer ikke en korrekt rangering, og en korrekt rangering garanterer ikke, at brugerne forstår rapporten. I har derfor både en teknisk og menneskecentreret evalueringsplan.

**Fremragende eksamensformulering:** »Vi gjorde succes operationel ved at formulere tre adskilte mål, der kræver hver sin type evidens.«

## A2. Projektets dokumenterede beslutningshistorik

En forsvarlig fortælling skal skelne mellem det, I *først forestillede jer*, og det, I *endte med at implementere*:

1. **Undersøgelse af datasæt.** To Kaggle/LinkedIn-snapshots og O*NET blev inspiceret i `preprocessing/data_inspection_primary.ipynb` og `secondary.ipynb`.
2. **Afgrænsning til navngivne værktøjer.** Projektets definition af *skill* blev indsnævret; bløde færdigheder og brede metoder som »teamwork« og »machine learning« indgår ikke i hovedresultatet.
3. **Scope-cut: ingen tids-trends.** Ifølge README og TODO viste januar/april-sammenligningen scraperforskelle frem for solidt grundlag for trends. I erstattede trends med **robusthed af rangeringerne**.
4. **Fælles matcher.** Logikken fra de to inspection-notebooks blev samlet i `skillradar/matcher.py` for at bruge samme normalisering af CV'er og opslag.
5. **Kvalitetsfejl blev korrigeret.** Deduplikering af jobopslag, UX-rollepattern og tvetydige aliaser blev forbedret på baggrund af data og fejlundersøgelse.
6. **Evaluering af alternativer.** I sammenlignede exact/vendor/acronym/full og dernæst keyword, LLMOnly, Hybrid og HybridVerify.
7. **Pilot → redesign.** To tidlige tilbagemeldinger førte til ændret rapport, forklaring af konfidensintervaller og radar-visualisering.
8. **Åben status pr. 4/11.** Peer trial med 6–8 studerende var endnu planlagt; matcher v2 og resterende forbedringer stod i TODO.

Dette forløb illustrerer *iterativ udvikling*: hypoteser, undersøgelser, målinger og ændringer frem for én fast plan fra første dag.

## A3. Beslutning: fravalg af trends over tid

**Problemet.** I havde to snapshots af LinkedIn-opslag fra forskellige scrapers og med forskellig geografi/indsamling. Ændringer mellem snapshots kan skyldes sammensætning af data snarere end tidsudvikling i arbejdsgivernes behov.

**Metodisk risiko:** Confounding. Hvis datasæt A hovedsageligt inkluderer seniorstillinger, mens datasæt B indeholder flere entry-level-stillinger, kan en forskel i fx Docker blive forvekslet med et trend-skift.

**Jeres svar:** Goal 2 handler om, hvorvidt **rangordenen af værktøjer** er nogenlunde reproducerbar i uafhængige kilder, ikke om efterspørgslen steg fra januar til april.

**Hvorfor er det stærkt?** Det viser, at I lod datasættenes faktiske egenskaber styre, hvilke påstande I kunne forsvare.

**Alternativ i et større projekt:** Løbende indsamling med samme scraper, defineret population, ens lande, ens rolleudvælgelse og observationer over mange måneder; først derefter kan egentlig tidsanalyse diskuteres.

**Eksamenssvar:** »Vi opgav den mest spændende visualisering, fordi vores data ikke kunne identificere den effekt, vi ønskede at vise.«

In [ ]:
# Pædagogisk eksempel: en forskel mellem grupper er ikke nødvendigvis en tids-trend
import pandas as pd
example = pd.DataFrame({
    "snapshot": ["Jan", "Jan", "Apr", "Apr"],
    "niveau": ["entry", "senior", "entry", "senior"],
    "andel_Docker": [0.20, 0.60, 0.20, 0.60],
    "andel_af_opslag": [0.20, 0.80, 0.80, 0.20]
})
example["bidrag"] = example["andel_Docker"] * example["andel_af_opslag"]
print(example.groupby("snapshot")["bidrag"].sum())
display(example)

**Forklaring:** Efterspørgslen er 20 % i entry- og 60 % i senior-gruppen i *begge* snapshots. Alligevel falder den samlede vægtede Docker-andel, når gruppesammensætningen ændres. Det er et eksempel på sammensætningseffekt, **ikke et resultat fra jeres egne datasæt**. Hvis censor spørger om årsagssammenhæng, er pointen, at to observerede andele ikke isolerer den underliggende årsag.

## A4. Beslutning: deduplikering af postings

**Dokumenteret i README/TODO:** 4.049 tekst-identiske annoncer i primary og 342 i secondary blev fjernet blandt de relevante rolleopslag ved indlæsning med `load_posting_tools()`. De kunne have forskellige posting-ID'er, men samme normaliserede tekst.

**Hvorfor betyder det noget?** Hvis samme arbejdsgiver poster én annonce til fem byer, får dens værktøjer ellers fem stemmer i jeres efterspørgselsmål.

**Præcis teknisk metode:** `text_hash()` i `demand.py` normaliserer whitespace/case, SHA-1-hasher teksten og forkorter hashen; ved indlæsning fjernes dubletter efter `(role, text_hash)`, undtagen tom hash. Det er præcise normaliserede dubletter, ikke fuzzy near-duplicates.

**Eksamensdilemma:** Fjernelse af »samme opslag« kan undervurdere antallet af faktiske ledige stillinger, hvis hver by faktisk er et separat job. Men jeres mål er andelen af *forskellige annoncetekster*, der nævner et værktøj, og ikke estimering af antal ledige stillinger.

**Muligt alternativ:** Rapportér resultater både med og uden deduplikering som sensitivitetstest.

In [ ]:
# Hvad sker der med demand, når en annonce tælles mange gange?
from collections import Counter
raw = [set(["Python", "SQL"])] * 5 + [set(["Docker"])]
unique = [set(["Python", "SQL"]), set(["Docker"])]
def demand(ads):
    k = Counter(t for row in ads for t in row)
    return {t: round(v / len(ads), 3) for t, v in k.items()}
print("Med fem kopier:", demand(raw))
print("Efter dubletfjernelse:", demand(unique))

**Forklaring:** SQL og Python går fra 5/6 til 1/2, når der kun regnes med én kopi af annonceteksten. Dette er et illustrativt, ikke empirisk, eksempel. Det viser, at datakvalitet kan ændre selve forskningsresultatet.

## A5. Beslutning: rettelse af UX-rollemapping

**Dokumenteret i `TODO.md` og `skillradar/demand.py`:** Den oprindelige regex ramte bogstavsekvenser som `ui` i *Building* og *Circuit* samt `ux` i *Luxury*. Ifølge README var omkring **31 %** af de tidligere klassificerede UX-opslag fejlagtige. I indførte `\b` (word boundaries) for at begrænse det.

**Hvorfor er dette vigtigt?** Alle målinger i Goal 2 afhænger af, at nævneren er opslag for *den rigtige rolle*. En fejl her kan gøre en tilsyneladende velberegnet confidence interval meningsløs: statistikken er formelt korrekt, men på en fejlklassificeret population.

**Eksamenssvar:** »Vi opdagede, at en valid talbehandling ikke hjælper, hvis inputkategorien er forkert. Derfor reviderede vi rollenavnets regex og genberegnede data.«

In [ ]:
import re
old = re.compile(r"ui.*design", re.I)
new = re.compile(r"\b(?:ux|ui|uxui|user experience)\b.*design", re.I)
for title in ["Building Design Engineer", "UI Designer", "UX Designer", "Circuit Design Engineer"]:
    print(f"{title:27} gammelt={bool(old.search(title))} nyt={bool(new.search(title))}")

**Forklaring:** Den gamle illustrative regex kan matche `ui` inde i *Building*. Den nye kræver en ordgrænse. Eksemplet fremviser fejlkategorien, men den gamle regex i eksemplet er en rekonstruktion og **ikke verificeret ordret som et tidligere commit**.

## A6. Beslutning: Google/Facebook, case og tvetydige aliaser

`matcher.py` bruger `MANUAL_DROP = {"Google", "Facebook"}`. TODO beskriver manuelle undersøgelser af teksteksempler: disse navne stod ofte som firmaer eller sociale medier, ikke softwareværktøjer. Derimod blev `C` og `Go` bevaret efter konkrete vurderinger.

**Tradeoff:** Hvis I fjerner for mange aliaser, falder recall; hvis I beholder tvetydige ord, falder precision. En forbedring på et enkelt måltal kan skade et andet.

**Hvorfor blev de ikke fjernet automatisk ved høj frekvens?** Et legitimt værktøj som Excel kan være ekstremt hyppigt. Høj frekvens er ikke bevis for falsk positiv.

**Kritisk nuance:** TODO nævner stadig et kendt problem med »Go further« som muligt falsk positivt i keyword-versionen. At have manuel aliasliste er derfor en forbedring, ikke garanti for semantisk forståelse.

# B. Testing, debugging og softwarekvalitet
## B1. Hvilke testtyper har I?

`pytest` bruges som automatisk testværktøj. `pyproject.toml` angiver `testpaths = ["tests"]`, så `uv run pytest` finder tests i mappen. Projektkoden bruger små syntetiske fixtures, så meget af testen **ikke** afhænger af store Kaggle-filer.

Forskellige testformål:

- **Enhedstest:** `aliases()`, `is_abbrev()`, `tokens()`, `is_generic()` giver forventede outputs.
- **Matcher-tests:** samme inputtekst skal give forventet sæt af værktøjer.
- **Regressionstests:** tidligere fundne problemer må ikke genopstå efter ændring.
- **Integrationsprægede tests:** parser/report/demand kombinerer input, transformation og output.
- **Mock/fake tests for LLM:** `llm_matcher.py` kan få en fake client, så API-adfærd og grounding kan testes uden ekstern model.

**Vigtigt:** Vi kan beskrive tests ud fra kodefilerne, men at tests *findes*, er ikke det samme som at en CI-pipeline er etableret eller alle tests lykkes på alle maskiner. Der er ikke verificeret en workflow-fil i `.github/workflows/` i denne gennemgang.

## B2. Sådan læser du en pytest-fixture

I `tests/test_matcher.py` definerer I en lille tabel med kolonnerne `Workplace Example`, `Hot Technology`, `In Demand`, inkl. Python, SQL, AWS, Power BI, R, Go, C++, Node.js og generiske entries.

```python
@pytest.fixture
def software():
    return pd.DataFrame(ROWS, columns=["Workplace Example", "Hot Technology", "In Demand"])

@pytest.fixture
def m(software):
    return KeywordMatcher(software, manual_drop=[])
```

**Linje for linje:**
1. `@pytest.fixture` betyder, at pytest kan klargøre objektet automatisk for tests, der tager det som argument.
2. `software()` leverer et lille, kendt O*NET-lignende datasæt.
3. `m(software)` opbygger en matcher ud fra tabellen.
4. `manual_drop=[]` slår standard-drop-listen fra i nogle isolerede tests, så man kan teste grundreglerne uden globalt filter.

**Hvorfor er det godt?** En test kan fejle præcist, hvis aliaslogikken ændres. Man undgår at være afhængig af et 5 GB datagrundlag.

In [ ]:
# Kørbar mini-regressionstest med assert
import re
def contains_exact_tool(text, name):
    return bool(re.search(r"(?<![A-Za-z])" + re.escape(name) + r"(?![A-Za-z])", text, re.I))
assert contains_exact_tool("I use SQL", "SQL")
assert not contains_exact_tool("MySQL is different", "SQL")
print("Begge simple assertions bestået")

**Forklaring:** `assert` stopper eksekveringen, hvis en forventning ikke er opfyldt. I pytest bliver sådanne assertions til automatiske tests med fejlrapport. Selve regex-funktionen her er et undervisningseksempel, **ikke** jeres matcherimplementering.

## B3. Test først, ret derefter – regression debugging

**Direkte anbefaling i jeres README/tests:** Hvis I finder et forkert match i reelle tekster, skriv sætningen som en test i `tests/test_matcher.py`, se testen fejle, ret `skillradar/matcher.py`, og kør testene igen.

Det ligner en *test-first regression workflow*:
1. Isolér minimal reproducerbar fejl.
2. Skriv expected input/output.
3. Se at den nuværende kode bryder forventningen.
4. Ret mindst muligt i produktionskoden.
5. Genkør hele testpakken.
6. **Genbyg posting-tabellerne**, hvis matcheren har ændret sig.

Det sidste punkt er afgørende: Hvis CV-siden bruger ny matcher, mens de gamle posting-tabeller er udtrukket med en gammel matcher, er de to sider ikke længere konsistente.

**Eksempel fra TODO:** Azure, EC2, Power BI, GIS og småbogstavs-`bash` optræder blandt planlagte v2-forbedringer. Status i TODO er *åbne punkter*, ikke dokumenteret færdige rettelser.

## B4. Forskellen mellem unit test, evaluering og brugertest

| Test | Spørgsmål | Typisk input | Eksempel på resultat |
|---|---|---|---|
| Unit test | Gør en funktion det tilsigtede? | Kontrolleret, syntetisk tekst | `SQL` matches, `Payroll software` filtreres |
| Evaluering af Goal 1 | Er metoden god på virkelige, ukendte opslag? | Blind-annoteret testsæt | Precision 0,97 på 30 testopslag |
| Robusthed Goal 2 | Gentages rangering i anden kilde? | To uafhængige jobdatasæt | Spearman ρ på roller |
| Brugerundersøgelse Goal 3 | Forstår studerende appen? | Deltagere og opgaver | Opgavegennemførelse, rating og kommentarer |

**Eksamen:** »Vi har tests på flere niveauer, og de svarer på forskellige spørgsmål. Ingen enkelt testtype dokumenterer hele systemets kvalitet.«

## B5. Hvorfor »frosne modeller« og dev/test-split er vigtige

`evaluation/GUIDELINES.md` angiver en fast stikprøve på **40 Data Scientist-opslag**, **10 dev** og **30 held-out test**, seed 42. Begge annotatorer læser teksten uden først at se modellens output. `matcher-v1-frozen` bruges til hovedresultaterne; LLM-opsætningen er dokumenteret i `evaluation/llm_frozen_settings.txt`.

**Dev:** Her må man lede efter fejl, teste aliaser og tune LLM-parametre.

**Test:** Bruges til den endelige rapporterede præstation; gentagen tilpasning efter testoutput giver optimistiske estimater (**test leakage**).

**Nuance:** Efter evalueringen kan man godt udvikle en matcher v2 til produktet, men den skal beskrives som en ny version. v1-resultater kan ikke ukritisk overføres til v2.

**Eksamenssvar:** »Vi valgte at fryse den evaluerede konfiguration, så resultaterne kunne reproduceres og ikke blev tilpasset testudvalget.«

# C. Human-Centred Design og brugertest
## C1. Scenarie, behov og designkrav

Jeres centrale scenarie: En studerende har et CV, ønsker fx en data scientist-stilling og vil identificere teknologier, der kan være relevante at lære eller fremhæve.

Fra scenariet til krav:
- **Upload** PDF/DOCX/TXT/MD uden at kræve manuel kopiering af hele CV'et.
- **Role** bruger en begrænset, forklarlig liste af målroller.
- **Check** lader brugeren fjerne forkerte genkendelser og tilføje oversete værktøjer.
- **Report** viser *både* det CV'et allerede nævner, og mangler prioriteret efter observeret demand.
- **Usikkerhed** vises i intervaller og med plain-language forklaring.
- **Privatliv**: rå CV-tekst gemmes ikke i appens egne outputfiler.

**Hvorfor human-centred?** Det endelige resultat skal ikke bare være en beregning. Brugeren skal kunne se, forstå og korrigere systemets antagelser.

## C2. Human in the loop: Check-trinnet

I `app.py`:
- `matcher.extract_tools(ss.cv_text)` finder værktøjer.
- `matcher.evidence(ss.cv_text)` finder tekststykker, der begrunder matchene.
- `st.checkbox(...)` lader brugeren bevare/fjerne hvert værktøj.
- `st.multiselect(...)` lader brugeren tilføje teknologier fra matchervokabularet.
- Den korrigerede liste bruges **før** `build_report()`.

**Hvad løser det?** Et keyword-match kan være forkert i kontekst, og CV-parseren kan overse noget. Brugeren får kontrol over sin repræsentation.

**Hvad løser det ikke?** Brugeren kan overse fejl, acceptere et vildledende eksempel eller tilføje et værktøj, vedkommende ikke reelt behersker. Listen er en *bekræftet læsning af CV'et*, ikke verificeret kompetenceniveau.

**Vigtigt designspørgsmål:** Er det nok at vise et citat for at få tillid? Nej. Forklaring kan også få brugere til at stole på forkerte resultater. TODO henviser selv til Bansal et al. om denne risiko.

In [ ]:
# Enkel model af Check-trinnets effekt på den endelige liste
found = {"Python", "SQL", "Go"}
removed = {"Go"}   # brugeren vurderer at "Go" var almindeligt engelsk
added = {"Power BI"}
confirmed = (found - removed) | added
print("Automatisk fundet:", sorted(found))
print("Efter brugerkorrektion:", sorted(confirmed))

**Forklaring:** Mængdeoperationer er pædagogisk parallel til appens `cv_tools`-beregning. Bemærk at brugeren i appen tilføjer fra matchervokabularet, ikke vilkårlig fritekst. Det kan begrænse rettelse af out-of-vocabulary-værktøjer.

## C3. Hvad ændrede pilotfeedback konkret?

**Dokumenteret i README/TODO, efter to tidlige pilotrespons:**

| Feedback | Implementeret respons | Begrundelse |
|---|---|---|
| »95 %-ranges er uklare« | Plain-language forklaring, visuel intervalmarkering | Usikkerhed skal kunne fortolkes uden statistikbaggrund |
| »Vis først de værktøjer jeg allerede har« | `Already on your CV` øverst, derefter missing | Gør rapporten mindre ensidigt negativ og tydeliggør relevans |
| »Udseendet føltes generisk« | Radar-konsol-stil, teal/amber, dækningsmeter | Tydeligere informationshierarki og sammenhængende visuel identitet |

**Fortolkning:** Dette viser en iteration, hvor faktisk brugerfeedback påvirkede interaktionsdesign.

**Hvad kan I ikke sige endnu?** To tilbagemeldinger beviser ikke, at redesignet *målbart* forbedrer brugervenlighed hos den ønskede målgruppe. Det kræver systematisk evaluering efter ændringerne.

## C4. Peer-trial protokollen – hvad er faktisk planlagt?

`evaluation/user_study.md` beskriver en plan med **6–8 studerende** efter piloten. Deltagerne udfører faste opgaver med *think-aloud*:

1. Upload CV og vælg målrolle.
2. Gennemgå og ret værktøjslisten.
3. Generér skill-gap-rapport.
4. Udpeg 1–2 værktøjer, som de ville prioritere, og forklar hvorfor.

Efterfølgende spørgeskema med seks Likert-items (1–5):
- Korrekt værktøjsidentifikation.
- Relevans af manglende værktøjer.
- Hjælp til at beslutte hvad der skal læres.
- Forståelse af anbefalingerne.
- Tillid til rapporten.
- Let at bruge.

Plus to åbne spørgsmål: »Hvad var uklart?« og »Hvad ville du forbedre?«.

**Status:** I `TODO.md` (4. november 2026) er peer trial endnu ikke gennemført. Der står rekruttering som åben opgave, selvom to pilot-feedbackrækker og en redesignrunde er dokumenteret. Den seneste status kan have ændret sig efter denne dato; denne notebook påstår ikke, at peer trial er afsluttet.

## C5. Hvordan kan I analysere en peer trial?

Et solidt Goal 3-resultat bør rapportere **nævnere**: ikke kun »5 brugere klarede opgaven«, men »5 ud af 7 gennemførte uden hjælp«. Tilsvarende bør ratinger være ledsaget af antal respondenter og evt. fordeling.

**Correction rate** kan måles på flere måder, som ikke bør forveksles:
- `removed / detected`: Andel af automatisk fundne værktøjer, som brugeren fravalgte (mulig proxy for false positives).
- `added`: Antal manuelt tilføjede værktøjer pr. CV (mulig proxy for false negatives).
- `completed without help / participants`: Task success.

**Kritisk nuance:** Brugerrettelser er ikke en automatisk objektiv gold standard. Brugere kan også tage fejl, og tilføjede værktøjer kan være uden for det, der faktisk står i CV'et.

**Persondata:** Målinger i studieappen angiver antal fundne/fjernede/tilføjede, ikke selve CV'ets tekst.

In [ ]:
# Illustrativ analyse af pilot/peer-trial – SYNTHETISKE observationer
import pandas as pd
sessions = pd.DataFrame({
 "participant": ["P01", "P02", "P03", "P04"],
 "detected": [8, 6, 10, 5], "removed": [1, 0, 2, 0],
 "added": [2, 1, 0, 1], "task_complete_without_help": [True, True, False, True],
 "trust": [4, 3, 3, 5]
})
sessions["removed_share"] = sessions["removed"] / sessions["detected"]
display(sessions)
print("Task success:", sessions["task_complete_without_help"].mean())
print("Samlet fjernede / samlet fundne:", sessions["removed"].sum()/sessions["detected"].sum())

**Forklaring:** De fire deltagere ovenfor er **opfundet til undervisning**. De illustrerer forskellige nævnere og hvorfor `mean(removed/detected)` ikke nødvendigvis er det samme som `sum(removed)/sum(detected)`. Den første vægter hver deltager ens; den anden vægter hver fundet teknologi ens.

# D. Privacy, GDPR og sikkerhed
## D1. Hvor rejser persondata gennem systemet?

**Normal app (`app.py`)**:
1. Brugerens fil uploades via Streamlit `st.file_uploader`.
2. `up.getvalue()` giver filens bytes.
3. `read_cv_bytes()` læser PDF/DOCX/TXT/MD uden at skrive CV'et til disk.
4. Udtrukket tekst lægges i `st.session_state` (`cv_text`).
5. Matcher, evidens og rapport beregnes ud fra denne tekst.
6. Rapporten downloades af brugeren efter ønske; appen logger ikke CV-teksten i sin egen proces.

**Testing app (`app_testing.py`)**: Gemmer kun aftalte evalueringsfelter ved samtykke. `skillradar/study_logging.py` afviser ukendte felter, så en udvikler ikke ved et uheld kan føje fx `cv_text` til studieformularens record uden at ændre whitelist.

**Afgrænsning:** »Vi skriver ikke CV'et til en lokal fil« er ikke det samme som at kunne garantere, at ingen platform, browser, hostingudbyder eller underliggende infrastruktur behandler midlertidige data. En fuld privatlivsvurdering kræver også hosting- og logindstillinger.

## D2. Hvad gør `study_logging.py` konkret?

- `FIELDS` definerer præcist tilladte kolonner: deltagerkode, tidspunkt, CV-type, rolle, datasæt, `detected/added/removed`, seks ratings, to fritekstsvar.
- `_row(record)` afviser `unexpected` feltnavne med `ValueError`.
- `MAX_TEXT = 1000` begrænser fritekstfelternes længde, men er **ikke en garanti for at de er anonyme**.
- `save_response(...)` skriver enten til et Google Sheet eller til lokal `data/user_study/responses.csv`.
- Lokal CSV ligger i en `gitignored` mappe; konfiguration til Google Sheets ligger i Streamlit secrets.

**Metodisk pointe:** Dataminimering er implementeret konkret. Det er mere troværdigt at forklare *hvilke kolonner* I gemmer end blot at sige »vi tager GDPR alvorligt«.

**Resterende risiko:** En deltager kan skrive navn, e-mail eller private oplysninger i »What was confusing?«. Det er derfor for optimistisk at kalde alle fritekstsvar fuldstændig anonymiserede alene fordi deltager-ID er en kode.

In [ ]:
# Undervisningseksempel: felthvidliste som beskyttelse mod utilsigtet lagring
ALLOWED = {"participant_id", "role", "detected", "removed", "added"}
record_ok = {"participant_id": "P01", "role": "data scientist", "detected": 6}
record_bad = {**record_ok, "cv_text": "Private CV details ..."}
def validate_keys(row):
    unexpected = set(row) - ALLOWED
    if unexpected:
        raise ValueError(f"Uventede felter: {sorted(unexpected)}")
    return True
print("Godkendt:", validate_keys(record_ok))
try:
    validate_keys(record_bad)
except ValueError as exc:
    print("Afvist:", exc)

**Forklaring:** Den faktiske `study_logging.py` har flere tilladte felter og mere validering end dette eksempel, men følger samme whitelist-princip. Whitelisting mindsker utilsigtet datalagring; det erstatter ikke adgangskontrol, sletningspolitik, samtykke eller databehandleraftaler.

## D3. Lokal keyword-matcher kontra ekstern LLM

**Normal app:** Bruger `KeywordMatcher`, som udfører tekstmatching i Python og ikke sender CV-indhold til en ekstern LLM-model.

**Eksperimentelt LLM-modul:** `skillradar/llm_matcher.py` anvender Gemini API i metodeevalueringen. Kommentarerne siger eksplicit, at CV-tekst **ikke** bør sendes via gratis API-tier; brug betalt API og samtykke, hvis CV'er overhovedet skal behandles eksternt.

**Hvorfor er forskellen vigtig?** Ved et eksternt API skifter dataforløbet: personlige CV-oplysninger deles med en tredjepart. Det kræver særskilt vurdering af formål, behandlingsgrundlag, deling, lagring, overførsel og information til deltagerne.

**Også vigtigt:** LLM-cache i `data/llm_cache/` kan indeholde citater fra input og er derfor gitignored. `gitignore` beskytter kun mod utilsigtet commit, ikke mod at filerne findes lokalt eller i backups.

## D4. GDPR: hvad kan I med rette sige?

**Dokumenteret teknisk:** Data-minimeret logning, CV-håndtering i session memory i den normale app, whitelist af evalueringsfelter, lokale filer uden for Git-repository, ikke-offentlig håndtering af API-nøgler og beskrevet frivilligt samtykke i brugerprotokollen.

**Ikke dokumenteret som fuld gennemført compliance:** Formelt fastlagt dataansvarlig/databehandler, opbevaringsperiode, procedurer for indsigt/sletning, vurdering af tredjelandsoverførsler, lovligt behandlingsgrundlag for alle processer, sikkerhedstest eller databehandleraftaler. Sig ikke »GDPR-compliant« som ubetinget påstand uden den dokumentation.

**Eksamenssvar:** »Vi har implementeret dataminimering og undgået at gemme CV-tekst. Vi anerkender, at GDPR-overholdelse også omfatter organisatoriske og juridiske foranstaltninger, som vores kode alene ikke dokumenterer.«

**Mulig forbedring:** Skriftlig data retention policy, sletteprocedure for testdata, oplysning om hosting, threat modelling og kontrol af logs.

## D5. Sikkerhed: HTML-escaping og secrets

`skillradar/ui.py` beskriver, at dynamisk tekst escapes med `html.escape`, før den sættes ind i HTML. Det er vigtigt, fordi appen også anvender `st.markdown(..., unsafe_allow_html=True)` til sit eget design.

**Trussel:** Hvis CV-tekst bliver indsat som rå HTML, kunne brugerindhold ændre visningen eller skabe script-/markup-injektion. Escaping betyder, at tekst med `<` og `>` bliver vist som tekst i stedet for at blive fortolket som HTML.

**Secrets:** README og `user_testing/README_TESTING.md` placerer Google service account credentials i Streamlit Secrets eller gitignored `.streamlit/secrets.toml`; Gemini-nøgle i `.env` eller miljøvariable.

**Nuance:** `gitignore` forhindrer nye standard-commits, men sletter ikke en hemmelighed, som allerede er pushed i historikken. Hemmeligheder i commits skal roteres og fjernes sikkert.

In [ ]:
import html
user_text = '<script>alert("test")</script> SQL & Python'
print("Rå tekst:", user_text)
print("HTML-escaped:", html.escape(user_text))

**Forklaring:** HTML-escaping gør vinkelparenteser og `&` til tekst-entiteter. Det er ét lag af beskyttelse mod HTML-injektion, ikke et generelt værn mod alle typer angreb.

# E. Deployment og reproducerbarhed
## E1. Projektets struktur er bevidst opdelt

- `main.py`: kommandolinje (`build`, `report`).
- `skillradar/`: genanvendelige moduler for parsing, matching, demand, rapportering, statistik, UI og studie-logning.
- `preprocessing/`: analyse-notebooks til EDA og tidlige eksperimenter.
- `evaluation/`: sampling, annotering, metodeevalueringsscripts og resultater.
- `analysis/`: Goal 2 statistik og resultater.
- `tests/`: automatiske tests.
- `app.py`: brugerprototype.
- `app_testing.py`: separat app med feedbacktrin.

**Hvorfor er det et software engineering-valg?** Det bliver muligt at teste `matcher.py` separat fra Streamlit-grænsefladen, genbruge den samme logik i CLI og app og køre statistik uden at genlæse de store datafiler.

**Designtradeoff:** `app_testing.py` er en kopi af appen frem for en fælles routing-løsning. TODO advarer om, at rettelser i én app skal kopieres til den anden. Det er et konkret vedligeholdelsesproblem og et naturligt refactoring-mål.

## E2. Dependencies: `pyproject.toml` og `uv`

Projektets `pyproject.toml` angiver:
- Python `>=3.14`.
- `pandas`, `numpy`, `streamlit`, `pdfplumber`, `python-docx`, `matplotlib`.
- `google-genai` og `gspread` til alternative LLM-/studieworkflows.
- `pytest` i development dependencies.
- `hatchling` build backend for pakken `skillradar`.

Typiske kommandoer fra README:

```bash
uv sync
uv run pytest
uv run python main.py build --source primary
uv run python main.py build --source secondary
uv run streamlit run app.py
uv run streamlit run app_testing.py
```

**Hvorfor `uv`?** Det hjælper med konsistent miljøstyring og installation. Projektet har også `uv.lock` ifølge deploymentvejledningen. En lockfil forbedrer version-reproducerbarhed, men garanterer ikke automatisk identisk hardware, data, eksterne API-svar eller modeltilgængelighed.

**Eksamen:** Forklar altid kommandoerne som en sekvens: opsæt miljø → test → byg data → start webapp.

## E3. Hvad er deployet, og hvad bruges hvor?

`user_testing/README_TESTING.md` dokumenterer et deploy-flow til **Streamlit Community Cloud**:

1. Push repository med nødvendige appfiler til `main`.
2. Vælg `app_testing.py` som entrypoint.
3. Gem service account-konfiguration som Streamlit Secrets.
4. Brug Google Sheets til persistente feedbackrækker, fordi Streamlit Clouds lokale disk ikke er varig.
5. Hold CV'er, rå jobdata, `.env` og credentials ude af Git.

**Dokumenteret opsætning vs. verificeret aktiv drift:** README indeholder en guide og TODO nævner to pilotfeedbackrækker i Google Sheet, men denne notebook har ikke foretaget en live deployment-test. Sig derfor ikke, at alle Cloud-flows og permissions er uafhængigt verificeret i dag.

**Hvorfor ikke bruge store rådata i Cloud?** Appen læser små `posting_tools_*.csv`-filer og O*NET-filen, som er de nødvendige forudberegnede artefakter.

## E4. Hvad betyder reproducerbarhed i netop jeres projekt?

Tre niveauer:

**Kode-reproducerbarhed:** Samme version af `matcher.py`, statistiske scripts, testkommandoer og dependencies.

**Data-reproducerbarhed:** Præcis samme Kaggle-snapshots, O*NET-version, deduplikering og jobtitelfiltre. En anden scrape giver ikke nødvendigvis samme tal.

**Eksperiment-reproducerbarhed:** Fast `evaluation/manifest.csv` (seed 42), dev/test-split, frosne keywordregler, LLM-modelnavn, prompt, few-shot-ID'er og linking-threshold samt evalueringskonfiguration.

**Vigtigt:** LLM-resultater caches for at mindske omkostninger og variation, men det er ikke i sig selv dokumentation for reproducerbar live-modeladfærd senere. En API-model kan ændres eller udgå.

**Eksempel:** `evaluation/llm_frozen_settings.txt` fastlåser til evalueringen `gemini-3.5-flash-lite`, linking-threshold `1.0` (embedding-linking fra), few-shot `post_003,post_005`. `llm_matcher.py` kan have en anden *default* model. Brug den **frosne evalueringsopsætning**, ikke default, når I omtaler målte resultater.

In [ ]:
# Et eksempel på, hvordan parametre dokumenteres i et eksperiment
experiment = {
 "experiment": "illustrativ_model_sammenligning",
 "seed": 42,
 "split": {"dev": 10, "test": 30},
 "matcher_version": "matcher-v1-frozen",
 "uses_future_test_labels_for_tuning": False
}
import json
print(json.dumps(experiment, indent=2, ensure_ascii=False))

**Forklaring:** Dette er et *illustrativt* manifest, ikke et fuldstændigt eksport af jeres faktisk anvendte versionsmetadata. Dokumentationen viser, hvorfor test-split og versionsnavn skal følge resultattabellerne.

## E5. Build artifacts er en del af modellen

En vigtig fælde ved deployment: Der er to modelanvendelser:
1. Offline matcher på jobopslag → lille posting→tools-tabel.
2. Online matcher på CV → liste, brugeren retter.

Hvis keywordregler ændres, men posting-tabellen ikke genbygges, anvendes **forskellige versioner** af matcheren på de to sider. Da jeres rapport forudsætter fælles kanoniske teknologier, bliver beregningsgrundlaget inkonsistent.

**Forbedring:** Gem matcherversion/hash som metadata sammen med `posting_tools_primary.csv`, og kontrollér ved appstart, at CV-matcheren bruger samme version. Det er et forslag; ikke dokumenteret som allerede implementeret.

# F. Kritisk vurdering af evidens og resultater
## F1. Hvad har I rent faktisk dokumenteret?

**Goal 1, på 30 held-out Data Scientist-opslag:**
- 204 manuelt labellede værktøjsforekomster på tværs af items.
- 47 af dem (23 %) er uden for jeres O*NET-baserede vokabular.
- Keyword (full): precision 0,97, recall(vocab) 0,94, recall(all) 0,73.
- Hybrid (lenient scoring): precision 0,97, recall(all) 0,95.
- HybridVerify: recall(all) 0,73; vurdering af keywordkandidater alene gjorde meget lidt.

**Goal 2:** Rangreplikation mellem to datakilder varierer efter rolle: ρ = 0,87 for data analyst og software engineer, 0,80 for data scientist, men 0,63 for UX designer. Der er færre sikre junior/senior-forskelle efter korrektion for multiple tests.

**Goal 3:** Redesign efter to pilotfeedbackrækker er dokumenteret. En fuld peer trial med 6–8 deltagere er ifølge TODO (4. november) endnu planlagt.

**Statusnuance:** `evaluation/results/goal1_cv_keyword.md` indeholder også en separat CV-evaluering på **4 CV'er**: full keyword precision 0,96, recall(vocab) 1,00 og recall(all) 0,77. Dette er et lille CV-sample, ikke en stor generaliserbar undersøgelse. README's tidligere statuslinje om »CVs pending« er derfor mindre opdateret end denne specifikke resultatfil.

## F2. Strict og lenient giver markant forskellige historier

**Dokumenteret i `goal1_test_strict.md` og `goal1_test_lenient.md`:**

| Matcher | Strict precision | Strict recall(all) | Lenient precision | Lenient recall(all) |
|---|---:|---:|---:|---:|
| Keyword full | 0,97 | 0,73 | 0,97 | 0,73 |
| LLMOnly | 0,78 | 0,73 | 0,98 | 0,92 |
| Hybrid | 0,77 | 0,75 | 0,97 | 0,95 |
| HybridVerify | 0,98 | 0,73 | 0,98 | 0,73 |

**Hvorfor?** Strict evaluering godkender kun træffere mod vokabularet og markerer out-of-vocabulary-navne som fejl. Lenient accepterer relevante navne, hvis de kan matches til de menneskelige OOV-labels via navne-normalisering.

**Hvad skal I sige?** »Hybrid er meget bedre til open-vocabulary-recall under vores lenient-protokol. Under strict scoring fremstår mange legitime OOV-fund som false positives. Resultaterne afhænger derfor af den valgte definition af korrekthed, som vi skal begrunde og vise åbent.«

**Ekstra kritik:** Lenient name matching kan give tvivlsomme ækvivalenser, så manuel gennemgang af krediterede navne er vigtig.

In [ ]:
# Sammenlign strict vs lenient – TAL FRA PROJEKTETS RESULTATFILER
resultater = pd.DataFrame({
 "matcher": ["Keyword", "LLMOnly", "Hybrid", "HybridVerify"],
 "strict_precision": [0.97, 0.78, 0.77, 0.98],
 "lenient_precision": [0.97, 0.98, 0.97, 0.98],
 "strict_recall_all": [0.73, 0.73, 0.75, 0.73],
 "lenient_recall_all": [0.73, 0.92, 0.95, 0.73],
})
resultater["recall_forskel_pp"] = (resultater["lenient_recall_all"] - resultater["strict_recall_all"])*100
display(resultater)

**Forklaring:** Alle tallene i denne tabel er hentet fra jeres dokumenterede resultatfiler. Det beregnede kolonnefelt viser ændringen i procentpoint mellem evalueringsdefinitionerne; det er **ikke** en ny model, der blev bedre under træning. Modellen er den samme; bedømmelseskriteriet ændres.

## F3. Hvorfor »precision 0,97« ikke betyder at appen er 97 % korrekt

Precision = andelen af fundne værktøjer på et bestemt *evalueringssæt*, som også står i annoteringerne. Det siger **ikke**:
- at parseren altid kan læse alle CV-formater;
- at alle relevante kompetencer står på CV'et;
- at datarepræsentationen svarer til det danske jobmarked;
- at brugerens rettelser altid er korrekte;
- at en bestemt skill-gap-recommendation fører til et job;
- at 97 % af alle anbefalinger er rigtige for alle fremtidige CV'er.

**Kritisk forskel:** Goal 1 måler extraction; Goal 2 måler jobopslagsandele; Goal 3 måler menneskelig brugbarhed. Der er ikke ét globalt »97 % korrekt«-mål for produktet.

## F4. Hvad betyder små stikprøver?

- Goal 1 har kun **30 held-out posting-items**, fra **én målrolle**. Det begrænser sikkerheden ved generalisering til andre brancher og CV'er.
- CV-evalueringen er på **4 CV'er** i den aktuelle resultatfil, som er endnu mindre.
- I Goal 2 er nogle roller og juniorsamples meget små; særligt ML engineer og UX designer.
- En 6–8 personers peer trial er velegnet til at finde brugerproblemer, men ikke til en påstand som »95 % af danske studerende finder appen nyttig«.

**Bedre studiedesign:** Flere roller og lande, større stratificerede testudvalg, uafhængige annotatorer, flere CV-formater og flere repræsentative brugere.

**Vigtigt:** Flere opslag fra den samme scraper afhjælper ikke automatisk udvælgelsesbias.

## F5. Udvælgelsesbias og datamismatch

**Fakta i README:** Kilderne er LinkedIn-snapshots fra 2024 og hovedsageligt USA. SkillRadar er rettet mod studerende, også i Danmark. Jobopslagene er ikke garanteret et tilfældigt sample af alle stillinger.

**Konsekvenser:**
- Sprog og navngivning af værktøjer kan variere.
- Dansk arbejdsmarked, uddannelsesforløb og jobtitler kan afvige.
- Mange opslag er seniorspecifikke, mens målgruppen er studerende.
- Et værktøj, som er hyppigt i LinkedIn-opslag, er ikke nødvendigvis en kausal kompetencefaktor for ansættelse.

**Forsvarlig formulering:** »Vores ranking er deskriptiv for vores valgte datasæt og regler. Vi tester reproducerbarhed på en anden kilde, men det skaber ikke fuld ekstern validitet.«

## F6. Hvad fortæller Spearman-resultaterne – og hvad fortæller de ikke?

Faktiske tal fra `analysis/results/goal2_output.txt`:

| Rolle | Spearman ρ (all-level comparison) | 95 %-bootstrapinterval |
|---|---:|---|
| Data analyst | 0,87 | 0,77–0,93 |
| Software engineer | 0,87 | 0,84–0,92 |
| Data scientist | 0,80 | 0,66–0,88 |
| ML engineer | 0,82 | 0,68–0,88, lille sample |
| UX designer | 0,63 | 0,46–0,72 |

**Spearman** måler overensstemmelse i *rangordenen* af værktøjer, ikke om de to datasæt har de samme absolutte procentandele.

For Data Scientist ligger ρ på det forhåndsdefinerede mål 0,80, men intervallet er bredt, så man bør ikke fremlægge målet som entydigt statistisk opfyldt under alle betingelser. Ved matchede erfaringsniveauer er ρ 0,76.

**Hvorfor UX er vigtigt:** Det viser, at modellen ikke fungerer lige overbevisende for alle roller. At rapportere svage resultater er et tegn på faglig troværdighed, ikke et problem der skal skjules.

## F7. Multiple testing og falske fund

Goal 2 sammenligner junior og senior for en række værktøjer. Hvis man tester 20–40 værktøjer på 5 %-niveau, kan nogle tests se signifikante ud alene ved tilfældighed. Derfor anvender `robustness.py` en Benjamini–Hochberg-korrektion af p-værdierne.

**Dokumenteret eksempel:** For data analyst findes SQL i 36,2 % af junioropslag mod 66,2 % af senioropslag. Forskellen er +30,0 procentpoint og består BH-korrektionen. Mange andre værktøjsforskelle gør ikke.

**Begrænsning:** Disse forskelle er ikke bevis for, at senioritet *forårsager* krav om SQL. Jobtekstlængde, organisationer og rolle-undergrupper kan være forskellige mellem grupperne.

## F8. Hvorfor ikke bare skifte til Hybrid i appen?

Jeres målte resultater taler for, at Hybrid har højere open-vocabulary-recall. Alligevel er appen keyword-baseret.

**Dokumenterede hensyn:**
- Hybrid: cirka **17,7 sekunder pr. posting** i den frosne evaluering, mod millisekunder for keyword-matching.
- Millionopslag i offline-processen gør API-baseret inference dyrt og tidskrævende.
- Personlige CV'er til ekstern model rejser andre privatlivskrav.
- Keyword er deterministisk, enkelt at debugge og genbruger O*NET som fælles vokabular.

**Kritisk modargument:** Hybrid giver dokumenteret bedre recall på evalueringssættet. Der er derfor en kvalitetsomkostning ved at bevare keyword som slutbrugerløsning. Check-trinnet hjælper, men kan ikke fuldstændigt erstatte en bedre extraction-metode.

**Mulig kompromisløsning:** Offline hybrid kun på anonymiserede/offentlige jobopslag, lokal keyword-CV-matching, eller LLM som valgfri funktion med eksplicit samtykke. Det er fremtidige forslag, ikke implementeret hovedapp.

## F9. Hvad ville vi gøre anderledes med mere tid?

En prioriteret plan, tydeligt adskilt fra allerede gennemført arbejde:

| Prioritet | Tiltag | Begrundelse | Hvordan succesen måles |
|---|---|---|---|
| 1 | Afslut peer trial | Goal 3 mangler stærkere evidens | Task success, forståelse, trust, rettelsesrate |
| 2 | Udvid CV-/rolle-evaluering | Få testitems og én postingsrolle | P/R/F1 på flere CV'er, formater og roller |
| 3 | Matcher v2 med dokumenterede fejlfixes | OOV, Azure, EC2, Go | Foruddefineret ny test og regression |
| 4 | Danmark-relevant datasæt og evt. ESCO | Domænemismatch | Replikation på dansk datagrundlag |
| 5 | Pipeline-versionering | Konsekvens mellem offline/online matcher | Metadata, checks ved appstart |
| 6 | Forbedret privacy-governance | Klarere databehandling | Retention, adgang, slettepolitik, dokumentation |

**Eksamen:** Beskriv forbedringer som eksperimentelle hypoteser, ikke som garantier for øget kvalitet.

# G. Mundtlig eksamen – dybdegående spørgsmål med svar
## G1. 20 spørgsmål eksaminator kan stille

**1. Hvorfor var det nødvendigt at ændre projektets retning?**  Fordi de to kilder ikke gav et troværdigt grundlag for tidstrends; robusthed af en aktuel snapshots-rangering var mere passende.

**2. Hvad er jeres tre mål, og hvorfor er de separate?** Extraction, robusthed af demand og brugbar prototype, som hver kræver en forskellig type evidens.

**3. Hvorfor delte I i offline build og online rapport?** For at undgå at genanalysere store tekstmængder ved hvert brugerklik.

**4. Hvad var UX-regex-fejlen?** `ui` kunne matche inde i almindelige ord som Building; ordgrænser reducerede falske rolletildelinger.

**5. Hvorfor fjerne identiske annoncer?** De ville ellers overvægtige enkelte tekster i andelen af opslag med et værktøj.

**6. Hvorfor er QA-tests ikke nok til at vise en god extractor?** De tester forventninger i kontrollerede cases; performance på realtekst kræver et uafhængigt annoteret testudvalg.

**7. Hvad er regression testing?** Test der sikrer, at tidligere identificerede fejl ikke dukker op igen efter ændringer.

**8. Hvorfor fryse matcheren før test?** For at undgå leakage og bevare troværdige generaliseringsestimater.

**9. Hvorfor kontrollere fundne tools manuelt i appen?** Brugeren skal kunne korrigere falske positive og oversete matches; det forbedrer kontrol og forklarbarhed.

**10. Hvorfor viser I evidenscitater?** For at brugeren kan se det tekstgrundlag, matcheren brugte; men citater kan også skabe overdreven tillid.

**11. Hvorfor ændrede I rapporten efter pilotfeedback?** Brugerne havde svært ved ranges, prioriterede allerede kendte værktøjer og ønskede tydeligere visuel struktur.

**12. Hvad dokumenterer to pilotrespons?** Problemfund og en designiteration, ikke populationsbred dokumentation af øget brugbarhed.

**13. Hvorfor gemmer I ikke CV'er i studieloggen?** Dataminimering og mindre risiko for spredning af personoplysninger.

**14. Kan I love fuld GDPR-compliance?** Ikke ud fra koden alene; der mangler dokumentation for juridiske/organisatoriske forhold.

**15. Hvad gør `uv sync`?** Installerer projektets afhængigheder i et konsistent miljø, baseret på projektkonfiguration og lockfile.

**16. Hvad gør I, hvis matcheren ændres efter build?** Genbyg behandlet posting-data, så CV og postings matches efter samme regler.

**17. Hvorfor skifter strict/lenient-resultater?** Fordi scoringens definition af korrekt OOV-værktøj ændrer sig, ikke fordi modellerne trænes om.

**18. Hvorfor kan I ikke sige at data er repræsentative for Danmark?** Kilderne er 2024 LinkedIn-snapshots med overvægt af USA.

**19. Hvorfor er Goal 1-resultatet ikke et produktmål for alle roller?** Det målte testsæt har kun 30 Data Scientist-postings og meget få CV'er.

**20. Hvilket kompromis er det vigtigste?** Højere open-vocabulary-recall med hybrid versus hastighed, forklarbarhed, privatliv og implementeringsomkostning i keyword-appen.

## G2. Øvelse: Få alle forklaringer til at hænge sammen

Øv følgende kæde uden noter:

1. **Problem:** Studerende ved ikke, hvilke teknologier det kan være relevant at fremhæve eller lære.
2. **Data:** To historiske LinkedIn-snapshots og O*NET som fælles teknologivokabular.
3. **Forberedelse:** Fem rollemønstre, tekstbaseret extraction, hashes/deduplikering, små behandlede CSV-tabeller.
4. **App:** Upload → vælg rolle → bekræft værktøjer → rapport.
5. **Beregning:** Andel opslag pr. værktøj + Wilson-intervaller + demand-vægtet coverage.
6. **Metodeevaluerering:** Menneskelige gold labels, P/R/F1, ablation, LLM/hybrid og strict/lenient.
7. **Robusthed:** Replikation via Spearman, bootstrap rank-intervaller og junior/senior.
8. **HCD:** Pilotfeedback → redesign, og planlagt peer trial for brugbarhed.
9. **Begrænsning:** US-snapshot, OOV, små samples, ikke en løbende trendmåler.
10. **Næste skridt:** Bedre evaluering, v2 og lokalt relevant data.

Prøv at forklare hvert trin med præcis **én kodefil**, ét designvalg og én begrænsning.

## G3. En samlet, mundtlig forklaring (ca. 4–5 minutter)

> »Vores mål var at hjælpe studerende med at forstå, hvilke konkrete softwareværktøjer jobopslag nævner, og hvilke af dem der mangler i deres CV. Vi opdeler projektet i tre mål: pålidelig extraction, robust sammenligning med jobmarkedet og en forståelig webapp. Det er vigtigt, fordi en præcis extractor ikke automatisk giver en nyttig anbefaling.
>
> Vi startede med to LinkedIn-datasæt og O*NET. Data inspection viste, at vi ikke havde en god løbende tidsserie, så vi fravalgte egentlige trendpåstande. I stedet beregner vi efterspørgselsandele og tester, om rangeringerne ligner hinanden på tværs af uafhængige datasæt. Vi filtrerer opslag til fem målroller med regex, normaliserer erfaringsniveauer og fjerner dublerede annoncetekster.
>
> Vi samlede matchinglogikken i én O*NET-baseret keyword-matcher, der bruges både på CV'er og jobopslag. Den kan udtrække for eksempel SQL, Python og Power BI, også når navne skrives forskelligt. Under arbejdet opdagede vi konkrete problemer med UX-rollemapping og tvetydige ord som Google, som vi rettede ud fra teksteksempler og tests.
>
> Den tunge analyse af jobopslagene udføres offline og gemmes som små tabeller, så Streamlit-appen kan reagere hurtigt. I appen uploader brugeren et CV, vælger en rolle og retter den fundne teknologiliste. Det er et human-in-the-loop-design, fordi automatisk keyword-matching ikke altid forstår kontekst.
>
> Vi evaluerede ekstraktionen på 30 hold-out Data Scientist-jobopslag. Keyword-metoden opnåede precision 0,97 og recall 0,94 på værktøjer i vokabularet, men recall 0,73 når vi medregner alle menneskeligt annoterede teknologier. Hybrid fandt 0,95 af alle værktøjerne under vores lenient scoring. Derfor er der et reelt kompromis: Hybrid finder mere, mens keyword er langt hurtigere og holder CV-behandling væk fra eksterne LLM-tjenester.
>
> I Goal 2 fandt vi en Spearman-korrelation på 0,87 for data analyst og software engineer, 0,80 for data scientist, men kun 0,63 for UX designer. Det viser rimelig robusthed i nogle roller, men ikke alle. Vi viser også usikkerhed og undgår at påstå, at disse historiske amerikanske LinkedIn-opslag repræsenterer alle danske jobs.
>
> Til sidst brugte vi tidlig feedback fra pilotbrugere til at redesigne rapporten: tydeligere intervaller, fremhævelse af det CV'et allerede indeholder og en radar-visualisering. Den mere omfattende peer trial stod endnu som planlagt i vores status fra 4. november. Fremtidigt arbejde er bl.a. en større CV-evaluering, en ny matcher-version og mere Danmark-relevante data. Vores vigtigste læring er, at godt systemdesign også handler om kvaliteten af datagrundlaget, evalueringsdefinitioner og brugerens mulighed for at forstå og rette resultaterne.«

## G4. Eksamenscheckliste – markér mentalt hvad du kan svare på

- [ ] Jeg kan forklare forskellen mellem de **tre goals**.
- [ ] Jeg kan fortælle **hvorfor trendanalysen blev fravalgt**.
- [ ] Jeg kan forklare **UX regex-fejlen** og dens konsekvenser.
- [ ] Jeg kan forklare **dubletfjernelse** og hvordan den ændrer demand.
- [ ] Jeg kan vise en **pytest-fixture** og en **regressionstest**.
- [ ] Jeg ved hvorfor det er forkert at bruge **testsplit til tuning**.
- [ ] Jeg kan forklare **human in the loop** og begrænsningerne ved det.
- [ ] Jeg kan skelne mellem **pilotfeedback** og en fuldt gennemført peer trial.
- [ ] Jeg kan redegøre for **hvilke felter der gemmes**, og hvordan CV-data håndteres.
- [ ] Jeg kan forklare forskellen mellem **privacy-praksis og dokumenteret GDPR-compliance**.
- [ ] Jeg kan forklare `uv sync`, `build` og Streamlit-deployment.
- [ ] Jeg ved hvorfor **posting-tabellerne skal genbygges**, når matcheren ændres.
- [ ] Jeg kan præsentere **strict og lenient** side om side.
- [ ] Jeg kan kritisere ekstern validitet, OOV-begrænsning og små samples.
- [ ] Jeg kan formulere **fremtidigt arbejde uden at påstå, at det er implementeret**.

# Kildeliste – præcise GitHub-filer

Projekt: https://github.com/Lucialululu/SkillRadar

- [`TODO.md`](https://github.com/Lucialululu/SkillRadar/blob/main/TODO.md) – udviklingshistorik, beslutninger, seneste åbne punkter og resultater (4. nov. 2026).
- [`README.md`](https://github.com/Lucialululu/SkillRadar/blob/main/README.md) – de tre goals, kilder, installation, test og scope.
- [`tests/test_matcher.py`](https://github.com/Lucialululu/SkillRadar/blob/main/tests/test_matcher.py) – konkrete test-fixtures og regressionstests.
- [`skillradar/matcher.py`](https://github.com/Lucialululu/SkillRadar/blob/main/skillradar/matcher.py) – aliasregler, case, MANUAL_DROP, evidence.
- [`skillradar/demand.py`](https://github.com/Lucialululu/SkillRadar/blob/main/skillradar/demand.py) – rollemønstre, hashing, build og deduplikering.
- [`app.py`](https://github.com/Lucialululu/SkillRadar/blob/main/app.py) – normal app og session state.
- [`app_testing.py`](https://github.com/Lucialululu/SkillRadar/blob/main/app_testing.py) – brugerundersøgelsesversion.
- [`skillradar/ui.py`](https://github.com/Lucialululu/SkillRadar/blob/main/skillradar/ui.py) – HTML/CSS og escaping.
- [`skillradar/study_logging.py`](https://github.com/Lucialululu/SkillRadar/blob/main/skillradar/study_logging.py) – validerede studie-felter og CSV/Google Sheets.
- [`evaluation/user_study.md`](https://github.com/Lucialululu/SkillRadar/blob/main/evaluation/user_study.md) – protokol, samtykke og spørgsmål.
- [`evaluation/GUIDELINES.md`](https://github.com/Lucialululu/SkillRadar/blob/main/evaluation/GUIDELINES.md) – blind annotation, dev/test og gold.
- [`evaluation/results/goal1_test_strict.md`](https://github.com/Lucialululu/SkillRadar/blob/main/evaluation/results/goal1_test_strict.md) og [`goal1_test_lenient.md`](https://github.com/Lucialululu/SkillRadar/blob/main/evaluation/results/goal1_test_lenient.md) – sammenlignelige resultattabeller.
- [`evaluation/results/goal1_cv_keyword.md`](https://github.com/Lucialululu/SkillRadar/blob/main/evaluation/results/goal1_cv_keyword.md) – lille CV-evaluering.
- [`analysis/results/goal2_output.txt`](https://github.com/Lucialululu/SkillRadar/blob/main/analysis/results/goal2_output.txt) – faktiske rang-/senioritetsresultater.
- [`evaluation/llm_frozen_settings.txt`](https://github.com/Lucialululu/SkillRadar/blob/main/evaluation/llm_frozen_settings.txt) – eksakt frozen LLM-konfiguration.
- [`pyproject.toml`](https://github.com/Lucialululu/SkillRadar/blob/main/pyproject.toml) – Python og afhængigheder.
- [`user_testing/README_TESTING.md`](https://github.com/Lucialululu/SkillRadar/blob/main/user_testing/README_TESTING.md) – deployment og Google Sheets.

**Statusdato:** Resultater og TODO kan ændre sig i GitHub efter 5. november 2026. Denne notebook beskriver den gennemgåede tilstand og gør klart, hvor der er planer frem for dokumenterede resultater.

**Afgrænsning:** Kodeøvelserne er undervisningseksempler, ikke nye evalueringer eller faktiske peer-trial-data.